In [1]:
import re, math
from collections import Counter

report = """# Annual Report

## Software Engineering
In Q4 the team resolved INC-2023-Q4-011, a deployment outage. Engineers fixed 247 bugs and shipped four releases.

## Cybersecurity Analysis
Incident INC-2023-Q4-011 involved a phishing attempt. The security team contained it within two hours and updated policies.

## Financial Analysis
Revenue grew 12% year over year. Operating costs stayed flat and the company ended the year with a healthy cash reserve.

## Methodology
Data was collected from internal systems and reviewed quarterly by an independent team.
"""

def chunk_by_section(document_text):
    return re.split(r"\n## ", document_text)

## the BM25 index

In [7]:
STOP_WORDS = {"a", "an", "the", "and", "or", "of", "to", "in", "on", "with",
              "what", "which", "who", "is", "are", "was", "were", "it", "its",
              "for", "by", "at", "as", "this", "that", "happened"}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())
    return [w for w in words if w not in STOP_WORDS]

class BM25Index:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs, self.tokens = [], []

    def add_document(self, doc):
        self.docs.append(doc)
        self.tokens.append(tokenize(doc["content"]))

    def search(self, query, k=3):
        q_terms = tokenize(query)
        n = len(self.tokens)
        avg_len = sum(len(t) for t in self.tokens) / n
        df = Counter()
        for toks in self.tokens:
            for term in set(toks):
                df[term] += 1

        scored = []
        for doc, toks in zip(self.docs, self.tokens):
            tf = Counter(toks)
            score = 0.0
            for term in q_terms:
                if term not in tf:
                    continue
                idf = math.log(1 + (n - df[term] + 0.5) / (df[term] + 0.5))
                num = tf[term] * (self.k1 + 1)
                den = tf[term] + self.k1 * (1 - self.b + self.b * len(toks) / avg_len)
                score += idf * num / den
            scored.append((doc, score))
        return sorted(scored, key=lambda p: p[1], reverse=True)[:k]

In [8]:
chunks = chunk_by_section(report)

store = BM25Index()
for chunk in chunks:
    store.add_document({"content": chunk})

results = store.search("What happened with INC-2023-Q4-011?", 3)

for doc, score in results:
    print(round(score, 3), "\n", doc["content"][:200], "\n----\n")

0.787 
 Software Engineering
In Q4 the team resolved INC-2023-Q4-011, a deployment outage. Engineers fixed 247 bugs and shipped four releases.
 
----

0.787 
 Cybersecurity Analysis
Incident INC-2023-Q4-011 involved a phishing attempt. The security team contained it within two hours and updated policies.
 
----

0.0 
 # Annual Report
 
----



In [4]:
%pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
